# Notebook 01 — Data Cleaning

Reads `data/faers_raw.csv` and produces:
- `data/faers_clean.csv` — full cleaned dataset (528k rows)
- `data/faers_serious.csv` — serious-event subset for PRR/ROR analysis

In [25]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '..')

df = pd.read_csv('../data/faers_raw.csv', low_memory=False)
print('Raw shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())

Raw shape: (528000, 30)

Missing values:
report_id                   0
receive_date                0
year                        0
month                       0
quarter                     0
serious                     0
serious_flags          301106
is_fatal                    0
is_hospitalized             0
is_life_threat              0
is_disabling                0
reactions                   0
primary_reaction            0
reaction_outcomes           0
patient_recovered           0
num_reactions               0
suspect_drug                0
brand_name                  0
drug_route                  0
drug_indication             0
manufacturer                0
pharm_class                 0
num_drugs                   0
drug_count_category         0
patient_age_years      151509
age_group                   0
patient_sex                 0
patient_weight_kg      379923
country                     0
report_age_days             0
dtype: int64


## 1. Rename columns

In [26]:
df = df.rename(columns={
    'suspect_drug':    'drugname',
    'primary_reaction': 'reaction'
})
print('Columns:', df.columns.tolist())

Columns: ['report_id', 'receive_date', 'year', 'month', 'quarter', 'serious', 'serious_flags', 'is_fatal', 'is_hospitalized', 'is_life_threat', 'is_disabling', 'reactions', 'reaction', 'reaction_outcomes', 'patient_recovered', 'num_reactions', 'drugname', 'brand_name', 'drug_route', 'drug_indication', 'manufacturer', 'pharm_class', 'num_drugs', 'drug_count_category', 'patient_age_years', 'age_group', 'patient_sex', 'patient_weight_kg', 'country', 'report_age_days']


## 2. Standardise drug names and reactions

In [27]:
df['drugname'] = (
    df['drugname'].astype(str)
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .replace('NAN', np.nan)
)

df['reaction'] = (
    df['reaction'].astype(str)
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .replace('NAN', np.nan)
)

print('Sample drug names:', df['drugname'].value_counts().head(5).index.tolist())
print('Sample reactions:',  df['reaction'].value_counts().head(5).index.tolist())

Sample drug names: ['TOFACITINIB', 'RISPERIDONE', 'RIVAROXABAN', 'AVANDIA', 'ETANERCEPT']
Sample reactions: ['DRUG INEFFECTIVE', 'DEATH', 'PNEUMONIA', 'DRUG HYPERSENSITIVITY', 'MYOCARDIAL INFARCTION']


## 3. Build outcome code with correct priority order

Priority: Death > Life-threatening > Hospitalised > Disabled > Other

**Bug fixed from original:** original code checked `is_hospitalized` before `is_life_threat`,
which could mask the more serious outcome.

In [28]:
def map_outcome(row):
    if row.get('is_fatal', 0) == 1:
        return 'DE'
    elif row.get('is_life_threat', 0) == 1:   # fixed: LT checked before HO
        return 'LT'
    elif row.get('is_hospitalized', 0) == 1:
        return 'HO'
    elif row.get('is_disabling', 0) == 1:
        return 'DS'
    else:
        return 'OT'

df['outc_cod'] = df.apply(map_outcome, axis=1)

outcome_map = {
    'DE': 'Death',
    'HO': 'Hospitalised',
    'LT': 'Life-threatening',
    'DS': 'Disabled',
    'OT': 'Other'
}
df['outcome_label'] = df['outc_cod'].map(outcome_map)

print('Outcome distribution:')
print(df['outc_cod'].value_counts())

Outcome distribution:
outc_cod
OT    303157
HO    147716
DE     54301
LT     16960
DS      5866
Name: count, dtype: int64


## 4. Parse dates and derive year/quarter

In [29]:
df['report_date'] = pd.to_datetime(df['receive_date'], errors='coerce')
df['year']    = df['report_date'].dt.year
df['quarter'] = df['report_date'].dt.to_period('Q').astype(str)

print('Date range:', df['report_date'].min(), '→', df['report_date'].max())
print('Missing dates:', df['report_date'].isna().sum())

Date range: 2015-01-01 00:00:00 → 2025-12-31 00:00:00
Missing dates: 0


## 5. Numeric age column

In [30]:
df['age'] = pd.to_numeric(df['patient_age_years'], errors='coerce')
print('Age — missing:', df['age'].isna().sum(), '| median:', df['age'].median())

Age — missing: 151509 | median: 59.0


## 6. Tag biologic drugs

In [31]:
BIOLOGIC_KEYWORDS = [
    'ADALIMUMAB', 'ETANERCEPT', 'INFLIXIMAB', 'CERTOLIZUMAB',
    'GOLIMUMAB', 'TOCILIZUMAB', 'ABATACEPT', 'RITUXIMAB',
    'VEDOLIZUMAB', 'USTEKINUMAB', 'SECUKINUMAB', 'IXEKIZUMAB',
    'RISANKIZUMAB', 'GUSELKUMAB', 'NIVOLUMAB', 'PEMBROLIZUMAB',
    'ATEZOLIZUMAB', 'IPILIMUMAB', 'TRASTUZUMAB', 'BEVACIZUMAB',
    'DENOSUMAB', 'DUPILUMAB', 'OMALIZUMAB', 'MEPOLIZUMAB',
    'DINUTUXIMAB', 'BLINATUMOMAB',
]

pattern = '|'.join(BIOLOGIC_KEYWORDS)
df['is_biologic'] = df['drugname'].str.contains(pattern, na=False, regex=True)
print('Biologic reports:', df['is_biologic'].sum())

Biologic reports: 92775


## 7. Deduplicate on report-drug-reaction triple

In [32]:
before = len(df)
df = df.drop_duplicates(subset=['report_id', 'drugname', 'reaction'])
print(f'Rows before: {before:,} → after dedup: {len(df):,} (removed {before - len(df):,})')

Rows before: 528,000 → after dedup: 528,000 (removed 0)


## 8. Drop rows missing essential columns

In [33]:
before = len(df)
df = df.dropna(subset=['drugname', 'reaction', 'outc_cod'])
print(f'Rows after dropping nulls: {len(df):,} (removed {before - len(df):,})')

Rows after dropping nulls: 528,000 (removed 0)


## 9. Build serious-event subset

**Change from original:** original filtered only `DE`, `HO`, `LT`.
Added `DS` (disabled) — it is a FAERS serious-event flag and should be included
in PRR/ROR analysis per standard pharmacovigilance practice.

In [34]:
serious = df[df['outc_cod'].isin(['DE', 'HO', 'LT', 'DS'])].copy()

print('Outcome distribution (serious subset):')
print(serious['outc_cod'].value_counts())

Outcome distribution (serious subset):
outc_cod
HO    147716
DE     54301
LT     16960
DS      5866
Name: count, dtype: int64


## 10. Save

In [35]:
df.to_csv('../data/faers_clean.csv', index=False)
serious.to_csv('../data/faers_serious.csv', index=False)

print('Cleaning complete')
print('Clean shape:  ', df.shape)
print('Serious shape:', serious.shape)

Cleaning complete
Clean shape:   (528000, 35)
Serious shape: (224843, 35)
